# Updated SAR Analysis: Southern Appalachian Amphibians

Reanalysis of the nested and island species–area models from Stout, Jessee & McMeen (2025) using the [`sars`](https://pypi.org/project/sars/) Python library. The `sars` library fits SAR models via **nonlinear least squares** (NLS) in arithmetic space, rather than the log-linear OLS used in the original publication. NLS minimizes residuals in the species-count scale and assumes additive error, whereas log-linear OLS minimizes residuals in log-space and assumes multiplicative error (Tjørve & Tjørve 2021).

> **Original publication:** Stout, J.B., Jessee, L.D. & McMeen, J.N. (2025). Nested and island models for determining the species-area relationship of southern Appalachian amphibians. *Journal of North American Herpetology* 2025(1):1–7.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sars
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 120


## Data Loading

Two datasets from the paper's appendices: the nested model (n=7, concentric areas) and island model (n=26, individual sites).


In [ ]:
nested_df = pd.read_csv("data/nested.csv")
island_df = pd.read_csv("data/island.csv")

groups = ["AmphibianSpecies", "FrogSpecies", "SalamanderSpecies"]
labels = ["Amphibians", "Frogs", "Salamanders"]

def load_sars(df, col):
    """Write a temp CSV and load via sars.from_csv."""
    tmp = df[["Area", col]].copy()
    tmp.to_csv("/tmp/sars_tmp.csv", index=False)
    return sars.from_csv("/tmp/sars_tmp.csv", area_col="Area", species_col=col)

print(f"Nested: {len(nested_df)} sites, area range {nested_df['Area'].min():.4f} – {nested_df['Area'].max():,.0f} km²")
print(f"Island: {len(island_df)} sites, area range {island_df['Area'].min():.4f} – {island_df['Area'].max():,.0f} km²")

display(nested_df)
display(island_df)


## Power-Law Fits (S = cA^z)

### Nested Model


In [ ]:
nested_power = {}

for label, col in zip(labels, groups):
    data = load_sars(nested_df, col)
    fit = sars.sar_power(data)
    nested_power[label] = fit
    print(f"{label}:  c = {fit.params['c']:.4f},  z = {fit.params['z']:.4f},  R² = {fit.r_squared:.4f}")
    print(f"          AIC = {fit.aic:.2f},  AICc = {fit.aicc:.2f},  BIC = {fit.bic:.2f}")
    print()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, label in zip(axes, labels):
    fit = nested_power[label]
    sars.plot_fit(fit, log=True, ax=ax)
    ax.set_title(f"Nested — {label} (log-log)")
    ax.set_xlabel("log Area")
    ax.set_ylabel(f"log {label}")
plt.tight_layout()
plt.show()


### Island Model


In [ ]:
island_power = {}

for label, col in zip(labels, groups):
    data = load_sars(island_df, col)
    fit = sars.sar_power(data)
    island_power[label] = fit
    print(f"{label}:  c = {fit.params['c']:.4f},  z = {fit.params['z']:.4f},  R² = {fit.r_squared:.4f}")
    print(f"          AIC = {fit.aic:.2f},  AICc = {fit.aicc:.2f},  BIC = {fit.bic:.2f}")
    print()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, label in zip(axes, labels):
    fit = island_power[label]
    sars.plot_fit(fit, log=True, ax=ax)
    ax.set_title(f"Island — {label} (log-log)")
    ax.set_xlabel("log Area")
    ax.set_ylabel(f"log {label}")
plt.tight_layout()
plt.show()


## Cross-Group & Cross-Dataset Power Comparison


In [ ]:
rows = []
for label in labels:
    nf = nested_power[label]
    isf = island_power[label]
    rows.append({
        "Group": label,
        "Nested c": round(nf.params['c'], 4),
        "Nested z": round(nf.params['z'], 4),
        "Nested R²": round(nf.r_squared, 4),
        "Island c": round(isf.params['c'], 4),
        "Island z": round(isf.params['z'], 4),
        "Island R²": round(isf.r_squared, 4),
    })
display(pd.DataFrame(rows))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors_n = {"Amphibians": "red", "Frogs": "red", "Salamanders": "red"}
colors_i = {"Amphibians": "blue", "Frogs": "blue", "Salamanders": "blue"}

for ax, label, col in zip(axes, labels, groups):
    nf = nested_power[label]
    isf = island_power[label]

    # nested data + curve
    n_areas = nested_df["Area"].values
    n_species = nested_df[col].values
    ax.scatter(np.log10(n_areas), np.log10(n_species), color="red", marker="o", label="Nested", zorder=5)

    # island data + curve
    i_areas = island_df["Area"].values
    i_species = island_df[col].values
    ax.scatter(np.log10(i_areas), np.log10(i_species), color="blue", marker="x", label="Island", zorder=5)

    # fitted curves in log-log
    x = np.linspace(-3.5, 5.5, 200)
    ax.plot(x, np.log10(nf.params['c']) + nf.params['z'] * x, "r-",
            label=f"Nested z={nf.params['z']:.4f}")
    ax.plot(x, np.log10(isf.params['c']) + isf.params['z'] * x, "b--",
            label=f"Island z={isf.params['z']:.4f}")

    ax.set_title(label)
    ax.set_xlabel("log₁₀ Area (km²)")
    ax.set_ylabel("log₁₀ Species")
    ax.legend(fontsize=7)

plt.suptitle("Nested vs. Island — NLS Power Fits (log-log)", y=1.02)
plt.tight_layout()
plt.show()


## Alternative Model Comparison for Weak Fits

The island power model produces R² < 0.80 for all three groups, with salamanders notably weak. Compare power, linear, logarithmic, and powerR (3-parameter power) models for each island group.


In [ ]:
alt_models = [
    ("power",  sars.sar_power),
    ("linear", sars.sar_linear),
    ("loga",   sars.sar_loga),
    ("powerR", sars.sar_powerR),
]

island_alts = {}
for label, col in zip(labels, groups):
    data = load_sars(island_df, col)
    fits = {}
    rows = []
    for name, func in alt_models:
        try:
            fit = func(data)
            fits[name] = fit
            rows.append({"Model": name, "R²": round(fit.r_squared, 4),
                         "AIC": round(fit.aic, 2), "BIC": round(fit.bic, 2)})
        except Exception as e:
            rows.append({"Model": name, "R²": None, "AIC": None, "BIC": None})

    island_alts[label] = fits
    print(f"\n=== Island — {label} ===")
    display(pd.DataFrame(rows))


### Island Salamanders — Power vs. Linear Residuals

The linear model outperforms power for island salamanders. Examine residuals side by side.


In [ ]:
sal_data = load_sars(island_df, "SalamanderSpecies")
sal_power = island_alts["Salamanders"]["power"]
sal_linear = island_alts["Salamanders"]["linear"]

areas = island_df["Area"].values
obs = island_df["SalamanderSpecies"].values
pow_pred = sal_power.predict(areas)
lin_pred = sal_linear.predict(areas)

resid_df = pd.DataFrame({
    "Locality": island_df["Locality"],
    "Area": areas,
    "Observed": obs,
    "Power Pred": np.round(pow_pred, 1),
    "Power Resid": np.round(obs - pow_pred, 2),
    "Linear Pred": np.round(lin_pred, 1),
    "Linear Resid": np.round(obs - lin_pred, 2),
})
display(resid_df)

print(f"\nPower RMSE:  {np.sqrt(np.mean((obs - pow_pred)**2)):.2f}")
print(f"Linear RMSE: {np.sqrt(np.mean((obs - lin_pred)**2)):.2f}")


## Residual Diagnostics


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, (dataset_name, power_dict) in enumerate([("Nested", nested_power), ("Island", island_power)]):
    for j, label in enumerate(labels):
        sars.plot_residuals(power_dict[label], ax=axes[i, j])
        axes[i, j].set_title(f"{dataset_name} — {label}")

plt.tight_layout()
plt.show()


## Multi-Model Comparison

Fit all 20 SAR models supported by `sars` and rank by information criteria.

> **Note on AICc:** With n = 7 (nested) and n = 26 (island), AICc is finite for 2-parameter models in both datasets. For 3-parameter models on the nested data (n = 7, k = 4 including σ), AICc correction is large but still defined since n > k + 1. Results for 3-parameter models on small datasets should be interpreted cautiously.


### Nested — Multi-Model


In [ ]:
nested_multi = {}
for label, col in zip(labels, groups):
    data = load_sars(nested_df, col)
    multi = sars.sar_multi(data)
    nested_multi[label] = multi
    print(f"\n=== Nested — {label} (top 5 by BIC) ===")
    display(multi.summary[["model", "R2", "AIC", "AICc", "BIC", "shape"]].head(5))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, label in zip(axes, labels):
    sars.plot_multi(nested_multi[label], top_n=5, ax=ax)
    ax.set_title(f"Nested — {label}")
    ax.set_xlabel("Area (km²)")
    ax.set_ylabel(f"{label}")
plt.tight_layout()
plt.show()


### Island — Multi-Model


In [ ]:
island_multi = {}
for label, col in zip(labels, groups):
    data = load_sars(island_df, col)
    multi = sars.sar_multi(data)
    island_multi[label] = multi
    print(f"\n=== Island — {label} (top 5 by BIC) ===")
    display(multi.summary[["model", "R2", "AIC", "AICc", "BIC", "shape"]].head(5))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, label in zip(axes, labels):
    sars.plot_multi(island_multi[label], top_n=5, ax=ax)
    ax.set_title(f"Island — {label}")
    ax.set_xlabel("Area (km²)")
    ax.set_ylabel(f"{label}")
plt.tight_layout()
plt.show()


## Threshold Analysis

`sars.sar_threshold()` tests for breakpoints (small-island effect) by comparing continuous two-slope (ContOne), left-horizontal + right slope (ZslopeOne), and simple linear (no breakpoint) models.


In [ ]:
for dataset_name, df in [("Nested", nested_df), ("Island", island_df)]:
    print(f"\n{'=' * 50}")
    print(f"  {dataset_name} — Threshold Analysis")
    print(f"{'=' * 50}")
    for label, col in zip(labels, groups):
        data = load_sars(df, col)
        thresh = sars.sar_threshold(data)
        print(f"\n{label}: best = {thresh.best_model}, breakpoint = {thresh.best_breakpoint}")
        display(thresh.summary)


## Model Averaging

Model averaging uses Akaike weights derived from AICc to produce information-theoretic weighted predictions. This requires finite AICc for all models. With the island dataset (n = 26), this should be fully available. The nested dataset (n = 7) may have some models with AICc issues depending on parameter count.

We attempt model averaging for each dataset and group, skipping if AICc-based weights are all zero.


In [ ]:
import math

for dataset_name, df in [("Nested", nested_df), ("Island", island_df)]:
    print(f"\n{'=' * 50}")
    print(f"  {dataset_name} — Model Averaging")
    print(f"{'=' * 50}")
    for label, col in zip(labels, groups):
        data = load_sars(df, col)
        try:
            avg = sars.sar_average(data)
            # Check if weights are meaningful
            top_weights = sorted(avg.weights.items(), key=lambda x: -x[1])[:5]
            total_w = sum(w for _, w in top_weights)
            if total_w < 0.01:
                print(f"\n{label}: Akaike weights ≈ 0 (AICc limitation). Skipping.")
                continue
            print(f"\n{label} — Top 5 Akaike weights:")
            for model, w in top_weights:
                print(f"  {model:<12} {w:.4f}")

            sars.plot_average(avg)
            plt.title(f"{dataset_name} — {label} — Model-Averaged SAR")
            plt.xlabel("Area (km²)")
            plt.ylabel(f"{label}")
            plt.show()
        except Exception as e:
            print(f"\n{label}: Model averaging failed — {e}")


## Bootstrap Confidence Intervals

Bootstrap CIs require model averaging to produce meaningful weights. We attempt this for each group where model averaging succeeded.


In [ ]:
for dataset_name, df in [("Island", island_df)]:
    print(f"\n{'=' * 50}")
    print(f"  {dataset_name} — Bootstrap CI")
    print(f"{'=' * 50}")
    for label, col in zip(labels, groups):
        data = load_sars(df, col)
        try:
            avg = sars.sar_average(data)
            # check weights are non-trivial
            total_w = sum(avg.weights.values())
            if total_w < 0.01:
                print(f"\n{label}: Skipped (zero Akaike weights).")
                continue

            boot = sars.bootstrap_ci(data, n_boot=500, conf=0.95,
                                     rng=np.random.default_rng(42))
            sars.plot_average(avg, boot=boot)
            plt.title(f"{dataset_name} — {label} — Model-Averaged SAR with 95% Bootstrap CI")
            plt.xlabel("Area (km²)")
            plt.ylabel(f"{label}")
            plt.show()
            print(f"\n{label}: Bootstrap CI computed successfully.")
        except Exception as e:
            print(f"\n{label}: Bootstrap CI failed — {e}")


## Predictive Comparison: NLS vs. Original

Compare the NLS power model predictions against the original log-linear OLS predictions at several reference areas. The paper recommends averaging the nested and island models for practical use.


In [ ]:
from scipy import stats

# Original (log-linear) parameters
orig = {}
for label, col in zip(labels, groups):
    for dataset_name, df in [("nested", nested_df), ("island", island_df)]:
        log_a = np.log10(df["Area"].values)
        log_s = np.log10(df[col].values)
        result = stats.linregress(log_a, log_s)
        orig[f"{dataset_name}_{label}"] = {"C": 10**result.intercept, "z": result.slope}

test_areas = [1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0]

print("=== Amphibian Predictions: Nested ===")
print(f"{'Area (km²)':>12}  {'Orig':>8}  {'NLS':>8}  {'Diff':>6}")
print("-" * 40)
for a in test_areas:
    o = orig["nested_Amphibians"]
    n = nested_power["Amphibians"]
    orig_pred = o["C"] * a ** o["z"]
    nls_pred = n.params["c"] * a ** n.params["z"]
    print(f"{a:>12,.1f}  {orig_pred:>8.1f}  {nls_pred:>8.1f}  {nls_pred - orig_pred:>+6.1f}")

print("\n=== Amphibian Predictions: Island ===")
print(f"{'Area (km²)':>12}  {'Orig':>8}  {'NLS':>8}  {'Diff':>6}")
print("-" * 40)
for a in test_areas:
    o = orig["island_Amphibians"]
    n = island_power["Amphibians"]
    orig_pred = o["C"] * a ** o["z"]
    nls_pred = n.params["c"] * a ** n.params["z"]
    print(f"{a:>12,.1f}  {orig_pred:>8.1f}  {nls_pred:>8.1f}  {nls_pred - orig_pred:>+6.1f}")


In [ ]:
# Full comparison table — all groups, both datasets
rows = []
for label, col in zip(labels, groups):
    for dataset_name, df, power_dict in [("Nested", nested_df, nested_power),
                                          ("Island", island_df, island_power)]:
        o = orig[f"{dataset_name.lower()}_{label}"]
        n = power_dict[label]
        rows.append({
            "Dataset": dataset_name,
            "Group": label,
            "Orig C": round(o["C"], 2),
            "NLS c": round(n.params["c"], 4),
            "Orig z": round(o["z"], 4),
            "NLS z": round(n.params["z"], 4),
            "Orig R² (log)": "see analysis.ipynb",
            "NLS R² (arith)": round(n.r_squared, 4),
        })

display(pd.DataFrame(rows))


## References

- Rosenzweig, M.L. (1995). *Species Diversity in Space and Time*. Cambridge University Press.
- Drakare, S., Lennon, J.J. & Hillebrand, H. (2006). The imprint of the geographical, evolutionary and ecological context on species–area relationships. *Ecology Letters* 9:215–227.
- Stout, J.B., Jessee, L.D. & McMeen, J.N. (2025). Nested and island models for determining the species-area relationship of southern Appalachian amphibians. *Journal of North American Herpetology* 2025(1):1–7.
- Tjørve, E. & Tjørve, K.M.C. (2021). Mathematical expressions for the species–area relationship and the assumptions behind the models. In: Matthews, T.J., Triantis, K.A. & Whittaker, R.J. (Eds.), *The Species–Area Relationship: Theory and Application*. Cambridge University Press, pp. 157–184.


---

For full methodological discussion, see [`comparative_analysis.md`](comparative_analysis.md).
